[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sadriica/Curso_ANH/blob/main/modulo4_taller/modulo4_taller.ipynb)

Primera vez en Colab: ver la [guía](https://github.com/Sadriica/Curso_ANH/blob/main/guia_colab.md). Términos: [glosario](https://github.com/Sadriica/Curso_ANH/blob/main/glosario.md).

# Data & GIS para Energía
## Módulo 4: Aplicación. Reto. Taller.

El taller reúne los módulos anteriores en un flujo completo, con datos que llegan como en la vida
real: en distintos sistemas de coordenadas y en formatos distintos.

Objetivo: producir un mapa de idoneidad para un proyecto eólico en el norte de Colombia.

Pasos:

1. Cargar las fuentes (vienen en distintos CRS y formatos).
2. Unificar: llevar todo al mismo sistema de coordenadas.
3. Generar la malla (H3).
4. Mapear las fuentes sobre la malla.
5. Visualizar.
6. Decidir: combinar los criterios con AHP y obtener el mapa final.

Corre completo en Colab. Los archivos están también en `recursos/`.

## 0. Preparación

In [ ]:
!pip install -q geopandas "h3>=4.1" folium mapclassify

In [179]:
import pandas as pd
import numpy as np
import geopandas as gpd
import h3
import folium
import xarray as xr
from rasterstats import zonal_stats
from shapely.geometry import Polygon
from shapely.geometry import Point


## 1. Las fuentes (distintos CRS y formatos)

Dos fuentes de ejemplo:

- **Fuente A (recurso eólico y vías):** puntos en EPSG:9377 (metros, oficial de Colombia), con
  `viento_ms` y `dist_via_km`.
- **Fuente B (radiación solar):** una tabla en EPSG:4326 (grados lat/lon), con `radiacion`.

Aquí se generan; en un caso real se cargan con `read_file` o `read_csv`.

In [199]:
# cargar municipios de colombia
mnpios_col=gpd.read_file('recursos/MUNICPIOS_COLOMBIA')
mnpios_col.set_index('MpCodigo',drop=True,inplace=True)
# Cargar datos de índice de pobreza
df_ipm = pd.read_excel('recursos/IPM-municipal-valor.xlsx')
# Cargar datos de índice de desempeño institucional
df_idi = pd.read_csv(
    'recursos/desempeno_institucional/desempeno_institucional_final.csv',
    dtype={'codigo_dane': str},
    index_col=0
)

In [200]:
codigos = df_ipm['Código Municipio']
# Arreglo de los codigos
cods_arreglados = list()
for codigo in codigos:
    if len(str(codigo)) == 4:
        codigo_fix = '0' + str(codigo)
        cods_arreglados.append(codigo_fix)
    else:
        codigo_fix = str(codigo)
        cods_arreglados.append(codigo_fix)

# Actualizacion de los codigos
df_ipm['Código Municipio'] = cods_arreglados
# Cambio de indice
df_ipm.set_index('Código Municipio',drop=True,inplace=True)
# Georreferenciacion del indice de pobreza multidimensional
df_ipm['IPM'] = [float(x.replace(',','.')) for x in df_ipm['IPM']] 
mnpios_col['ipm'] = df_ipm['IPM']
# Georreferenciacion idi
mnpios_col['idi'] = df_idi['desempeno_municipal']

In [95]:
# Cargar datos de especies
gdf_especies = gpd.read_file('recursos/AMB-017-A/AMB-017-A.shp')
# Cargar datos de pendiente
tif_pendiente = xr.open_dataarray('recursos/AMB-007-AT/Pendientes.tif')
# Cargar velocidad del viento
tif_viento = xr.open_dataarray('recursos/Velocidad_100m_patched.tif')

In [188]:
# Municipios de interés
base = pd.DataFrame({
    "sitio": ["Uribia", "Riohacha", "Maicao", "Manaure",
              "Valledupar", "Aguachica", "Santa Marta", "Barranquilla"],
    "lat": [11.71, 11.55, 11.38, 11.78, 10.46, 8.31, 11.24, 10.96],
    "lon": [-71.98, -72.91, -72.24, -72.44, -73.25, -73.63, -74.20, -74.80],
})

# Creación de Geodataframe
mnpios = gpd.GeoDataFrame(
    base,
    geometry=gpd.points_from_xy(base["lon"], base["lat"]), crs="EPSG:4326",
)


A cada una de las fuentes chequear su sistema de coordenadas.
Describir sus fuentes coordenadas
¿ Se puede trabajar con los datos cómo se encuentran?

In [98]:
crs_especies = gdf_especies.crs
crs_pendiente = tif_pendiente.rio.crs
crs_viento = tif_viento.rio.crs
crs_idi_idm = mnpios_col.crs

crs_municipios = mnpios.crs
# Revision de las fuentes de los datos
print(f'Sistema coordenado especies aves:{crs_especies}')
print(f'Sistema coordenado pendiente:{crs_pendiente}')
print(f'Sistema coordenado viento:{crs_viento}')
print(f'Sistema coordenado especies municipios interes:{crs_municipios}')
print(f'Sistema coordenado especies IDI, IPM:{crs_idi_idm}')


Sistema coordenado especies aves:EPSG:9377
Sistema coordenado pendiente:PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1329"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","20046"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",4],PARAMETER["central_meridian",-73],PARAMETER["scale_factor",0.9992],PARAMETER["false_easting",5000000],PARAMETER["false_northing",2000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","9377"]]
Sistema coordenado viento:EPSG:4326
Sistema coordenado especies municipios interes:EPSG:4326
Sistema coordenado especies IDI, IPM:EPSG:9377


## 2. Unificar: mismo sistema de coordenadas

La Fuente A está en metros y la B en grados: no se pueden cruzar directamente. Se lleva todo a
EPSG:4326.

In [99]:
crs_reference = 'EPSG:4326'
# Modificacion de datos shape
gdf_especies = gdf_especies.to_crs(crs_reference)
mnpios_col = mnpios_col.to_crs(crs_reference)
# Modificacion de datos tif
tif_pendiente = tif_pendiente.rio.reproject(crs_reference)

In [102]:
# Corroborar que se hayan generado las modificaciones
print(f'Sistema coordenado especies aves: {gdf_especies.crs}')
print(f'Sistema coordenado especies IDI, IPM: {mnpios_col.crs}')
print(f'Sistema coordenado pendiente: {tif_pendiente.rio.crs}')

Sistema coordenado especies aves: EPSG:4326
Sistema coordenado especies IDI, IPM: EPSG:4326
Sistema coordenado pendiente: EPSG:4326


## 2.1 Generar la malla (H3)

Se debe extraer los datos por municipio para luego llevarse a la malla de h3
- fuente_a: velocidad del viento, pendiente y presencia de aves
- fuente_b: Índice de desempeño institucional e Índice de pobreza multidimensional

In [173]:
tif_viento.sel(y=11.71,x=-71.98,method='nearest').values[0]

np.float32(8.458921)

In [189]:
# Extraccion de datos para cada municipio
fuente_a = base.copy()
for idx in base.index:
    lat = base.iloc[idx,1]
    lon = base.iloc[idx,2]

    # Chequear tifs
    pendiente = tif_pendiente.sel(
        y=lat,x=lon,
        method='nearest'
    ).values[0]

    viento = tif_viento.sel(
        y=lat,x=lon,
        method='nearest'
    ).values[0]

    # Actualizacion especies
    punto = Point(lon, lat)	
    score_especies = gdf_especies[gdf_especies.intersects(punto)].iloc[0,0]
    # Actualización de los datos
    fuente_a.at[idx, 'vel_viento'] = viento
    fuente_a.at[idx, 'pendiente'] = pendiente
    fuente_a.at[idx, 'aves'] = score_especies

In [205]:
mnpios_col[mnpios_col['MpNombre'] == 'Riohacha']

,MpNombre,MpArea,MpNorma,MpCategor,MpAltitud,Restriccio,Depto,SHAPE_Leng,SHAPE_Area,cod_join,geometry,ipm,idi
MpCodigo,,,,,,,,,,,,,
44001,Riohacha,3082.410947,Ley 1766 de 2015,2,3,“No es apropiada su aplicación para la ubicaci...,La Guajira,331813.00519,3.082411e+09,44001,"POLYGON ((5011924.981 2835469.782, 5011985.063...",45.1,58.81


In [213]:
# Extraccion IDI e IPM
fuente_b = base.copy()

for idx,sitio in enumerate(base['sitio']):

    # busqueda del municipio
    mask = mnpios_col['MpNombre'] == sitio
    gdf_filtro = mnpios_col[mask]

    # extraccion informacion IDI e IPM
    idi = gdf_filtro.iloc[0,-1]
    ipm = gdf_filtro.iloc[0,-2]

    fuente_b.at[idx, 'idi'] = idi
    fuente_b.at[idx, 'ipm'] = ipm

In [215]:
# Creacion de geodataframe con las fuentes de datos
lats = base['lat']
lons = base['lon']
geo = gpd.points_from_xy(x=lons,y=lats)
fuente_a_4326 = gpd.GeoDataFrame(
    data=fuente_a,
    geometry=geo,
    crs=crs_reference
)
fuente_b_4326 = gpd.GeoDataFrame(
    data=fuente_b,
    geometry=geo,
    crs=crs_reference
)

## 3. Generar la malla (H3)

Se crea una malla hexagonal que cubre la zona de estudio. Cada celda tiene un código único y
tamaño casi igual, lo que hace comparables los sitios.

In [ ]:
RES = 5
zona = {"type": "Polygon", "coordinates": [[
    [-75.2, 8.0], [-71.5, 8.0], [-71.5, 12.2], [-75.2, 12.2], [-75.2, 8.0]]]}
malla = list(h3.geo_to_cells(zona, RES))   # el poligono va en orden GeoJSON [lon, lat]
print("celdas en la malla:", len(malla))

celdas en la malla: 472744


## 4. Mapear las fuentes sobre la malla

A cada sitio se le asigna su celda H3 y, por celda, se resumen los criterios con el promedio.
Así las dos fuentes quedan unidas en una sola tabla por celda.

In [ ]:
def celda(lat, lon):
    return h3.latlng_to_cell(lat, lon, RES)

# lat/lon de la fuente A ya reproyectada
fa = fuente_a_4326.copy()
fa["lat"] = fa.geometry.y; fa["lon"] = fa.geometry.x
fa["h3"] = [celda(la, lo) for la, lo in zip(fa["lat"], fa["lon"])]
fb = fuente_b_gdf.copy()
fb["h3"] = [celda(la, lo) for la, lo in zip(fb["lat"], fb["lon"])]

agg_a = fa.groupby("h3", as_index=False).agg(sitio=("sitio", lambda x: ", ".join(x)),
                                             viento_ms=("viento_ms", "mean"),
                                             dist_via_km=("dist_via_km", "mean"))
agg_b = fb.groupby("h3", as_index=False).agg(radiacion=("radiacion", "mean"))
celdas = agg_a.merge(agg_b, on="h3", how="outer")
celdas

## 5. Visualizar

Se dibujan las celdas con dato, coloreadas por viento, para verificar que la unificación y el
mapeo quedaron bien.

In [ ]:
import branca.colormap as cm
cmap = cm.LinearColormap(["blue", "yellow", "red"],
                         vmin=celdas["viento_ms"].min(), vmax=celdas["viento_ms"].max())
cmap.caption = "Viento (m/s)"
m = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in celdas.iterrows():
    borde = h3.cell_to_boundary(r["h3"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=cmap(r["viento_ms"]), fill_opacity=0.75,
                   tooltip=f"{r['sitio']}: viento {r['viento_ms']:.1f} m/s").add_to(m)
cmap.add_to(m)
m

## 6. Decidir: AHP y mapa de idoneidad

Con todo en la malla, se aplica AHP (Módulo 3) para combinar los criterios en un puntaje.
Criterios: viento (beneficio), distancia a vía (costo), radiación (beneficio).

In [ ]:
crit = {"viento_ms": "beneficio", "dist_via_km": "costo", "radiacion": "beneficio"}
def nz(col, sentido):
    v = celdas[col].astype(float); z = (v - v.min()) / (v.max() - v.min())
    return z if sentido == "beneficio" else 1 - z
norm = pd.DataFrame({c: nz(c, s) for c, s in crit.items()})

# pesos AHP (matriz de comparacion por pares, Modulo 3)
A = np.array([[1, 4, 2], [1/4, 1, 1/2], [1/2, 2, 1]], float)
w = (A / A.sum(0)).mean(1)
pesos = dict(zip(crit.keys(), w))
print("pesos AHP:", {c: round(v, 3) for c, v in pesos.items()})

celdas["idoneidad"] = sum(norm[c] * pesos[c] for c in crit)
celdas.sort_values("idoneidad", ascending=False)[["h3", "idoneidad"]].round(3).head()

In [ ]:
mapa = cm.LinearColormap(["blue", "yellow", "red"],
                         vmin=celdas["idoneidad"].min(), vmax=celdas["idoneidad"].max())
mapa.caption = "Idoneidad (AHP)"
m2 = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in celdas.iterrows():
    borde = h3.cell_to_boundary(r["h3"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=mapa(r["idoneidad"]), fill_opacity=0.8,
                   tooltip=f"{r['sitio']}: idoneidad {r['idoneidad']:.2f}").add_to(m2)
mapa.add_to(m2)
m2

### Guardar el resultado

El resultado se guarda en el formato de trabajo del proyecto: `.h3.parquet` (celda H3 e idoneidad).

In [ ]:
resultado = celdas[["h3", "sitio", "viento_ms", "dist_via_km", "radiacion", "idoneidad"]].rename(
    columns={"h3": "h3_index"})
resultado.to_parquet("resultado_idoneidad.h3.parquet", index=False)
print("guardado resultado_idoneidad.h3.parquet")
resultado.sort_values("idoneidad", ascending=False).round(3).head()

## Actividad individual

Modifique los pesos de la matriz AHP, por ejemplo dándole más importancia a la cercanía a vías,
y observe cómo cambia el mapa de idoneidad.

Resultado esperado: con los pesos actuales domina el viento, y Uribia y Manaure quedan arriba
pese a estar lejos de vías. Al subir el peso de la cercanía a vías, los sitios bien conectados
como Riohacha o Barranquilla escalan posiciones y los de la alta Guajira bajan. Con los mismos
datos y otra prioridad, el resultado cambia: por eso justificar los pesos es parte de la decisión.

## Cierre

Recorrido completo del taller:

1. Fuentes en distintos CRS y formatos.
2. Unificación a un mismo sistema de coordenadas.
3. Malla H3 sobre la zona.
4. Mapeo de las fuentes a la malla.
5. Visualización.
6. Decisión con AHP y mapa de idoneidad.

Es el mismo flujo del proyecto real, a pequeña escala. Para llevarlo más lejos: más criterios,
mayor resolución en la malla, exclusiones (zonas donde no se puede) y datos reales por municipio.